# 🏗️ InFoundry Architect - Cloud Training

Train a larger model (1.5B or 3B) on Google Colab's free T4 GPU.

**Requirements:**
- Enable GPU: Runtime → Change runtime type → T4 GPU
- Upload `generated_training_data.jsonl` when prompted

## Step 1: Install Dependencies

In [ ]:
# Install Oumi and dependencies
!pip install -q oumi transformers accelerate peft bitsandbytes
print("✅ Dependencies installed")

In [ ]:
# Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Upload Training Data

In [ ]:
from google.colab import files
print("Upload your generated_training_data.jsonl file:")
uploaded = files.upload()
print(f"\n✅ Uploaded: {list(uploaded.keys())}")

In [ ]:
# Verify data
import json
with open('generated_training_data.jsonl') as f:
    examples = [json.loads(line) for line in f]
print(f"✅ Loaded {len(examples)} training examples")
print(f"\nSample:")
print(json.dumps(examples[0], indent=2)[:500])

## Step 3: Configure Training

In [ ]:
# Training configuration
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Change to 3B if you have enough VRAM
OUTPUT_DIR = "./trained_model"
MAX_STEPS = 500
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 1e-5
LORA_R = 64
LORA_ALPHA = 128

print(f"Model: {MODEL_NAME}")
print(f"Training steps: {MAX_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")

## Step 4: Train the Model

In [ ]:
from oumi import train
from oumi.core.configs import TrainingConfig
from oumi.core.configs.params.model_params import ModelParams
from oumi.core.configs.params.training_params import TrainingParams, TrainerType
from oumi.core.configs.params.data_params import DataParams, DatasetParams, DatasetSplitParams
from oumi.core.configs.params.peft_params import PeftParams
from pathlib import Path

# Create training config
config = TrainingConfig(
    model=ModelParams(
        model_name=MODEL_NAME,
        trust_remote_code=True,
    ),
    training=TrainingParams(
        trainer_type=TrainerType.TRL_SFT,
        output_dir=OUTPUT_DIR,
        num_train_epochs=5,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        learning_rate=LEARNING_RATE,
        max_steps=MAX_STEPS,
        save_steps=100,
        logging_steps=25,
        use_peft=True,
        warmup_ratio=0.1,
    ),
    data=DataParams(
        train=DatasetSplitParams(
            datasets=[
                DatasetParams(
                    dataset_name="text_sft",
                    dataset_path=str(Path("generated_training_data.jsonl").absolute()),
                )
            ]
        )
    ),
    peft=PeftParams(
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
    ),
)

print("🚀 Starting training...")
print("="*60)
result = train(config)
print("="*60)
print("✅ Training complete!")

## Step 5: Test the Model

In [ ]:
from oumi import infer
from oumi.core.configs import InferenceConfig
from oumi.core.configs.params.generation_params import GenerationParams

# System prompt (same as training)
SYSTEM_PROMPT = '''You are an expert cloud architect. Given service profiles, recommend the optimal architecture.

You MUST respond with ONLY valid JSON in this EXACT format:
{
  "architecture": {
    "pattern": "serverless" or "microservices_ecs" or "kubernetes" or "event_driven" or "lift_and_shift",
    "components": ["api_gateway", "lambda", "ecs_cluster", "alb", "rds", "elasticache", etc.],
    "topology": "N services with PATTERN pattern on CLOUD",
    "scaling_strategy": "horizontal_autoscaling" or "serverless_autoscaling" or "kubernetes_hpa",
    "estimated_cost_tier": "low" or "medium" or "high",
    "rationale": "Brief explanation of why this architecture"
  },
  "inputs": {
    "service_count": NUMBER,
    "cloud_provider": "aws" or "gcp" or "azure"
  },
  "source": "ai_recommendation"
}

Do NOT include any text outside the JSON. Components must be strings, not objects.'''

# Configure inference
infer_config = InferenceConfig(
    model=ModelParams(
        model_name=MODEL_NAME,
        adapter_model=OUTPUT_DIR,
        trust_remote_code=True,
    ),
    generation=GenerationParams(
        max_new_tokens=400,
        temperature=0.1,
    ),
)

# Test inputs
test_inputs = [
    "Services: [api, auth, users], Language: python, DB: postgres, Cloud: aws, Latency p95: 200ms, Cost: $100/day",
    "Services: [api], Language: javascript, DB: dynamodb, Cloud: aws, Latency p95: 50ms, Cost: $20/day",
]

print("Testing trained model:")
print("="*60)

for user_input in test_inputs:
    prompt = f"{SYSTEM_PROMPT}\n\nUser Input: {user_input}\n\nJSON Response:"
    response = infer(infer_config, [prompt])
    result = str(response[0])
    
    # Clean up
    if "] metadata=" in result:
        result = result.split("] metadata=")[0]
    if "ASSISTANT:" in result:
        result = result.split("ASSISTANT:")[-1].strip()
    
    print(f"\nInput: {user_input[:50]}...")
    print(f"Output: {result[:500]}")
    print("-"*60)

## Step 6: Download Trained Model

In [ ]:
# Zip and download the trained model
!zip -r trained_model.zip ./trained_model

from google.colab import files
files.download('trained_model.zip')
print("\n✅ Model downloaded! Extract and use locally.")

## Next Steps

1. **Download** the `trained_model.zip` file
2. **Extract** it to your local `oumi/trained_model/` directory
3. **Update** `run_inference.py` to use `Qwen/Qwen2.5-1.5B-Instruct`
4. **Run** inference locally with the improved model!